# Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from plotly.offline import download_plotlyjs, init_notebook_mode, plot, iplot
import plotly.graph_objs as go
import plotly.express as px
import cufflinks as cf

init_notebook_mode(connected=True)  # For Notebooks

cf.go_offline() # For offline use

# Loading and inspecting Dataset

In [ ]:
df = pd.read_csv('/Users/festusattornelson/Documents/Projects/Python MSIT/car_data.csv')
df.head(2)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# Checkimg and dropping duplicate rows
df.duplicated().sum()

In [ ]:
#Checking for missing values
df.isna().sum()

In [ ]:
def initial_clean_data(data):
    """
      Cleans a given DataFrame by:
    - Removing duplicate rows
    - Standardizing text columns (trimming spaces, converting to lowercase)
    - datatype conversion (Year to datetime)
    - Drop missing values in Market Category
    - Filter data by year >= 1995

    Parameters:
    df (pd.DataFrame): The input DataFrame to be cleaned

    Returns:
    pd.DataFrame: The cleaned DataFrame

    """
    df = data.copy()

    #Remove duplicates
    df = df.drop_duplicates(keep='first').reset_index(drop=True)

    #Filter columns based on year >= 1995
    df = df.loc[df.Year >= 1995].reset_index(drop=True)

    #Drop missing values in Market Category
    #df['Market Category'] = df['Market Category'].replace("", np.nan)
    #df = df.dropna(subset=['Market Category']).reset_index(drop=True)

    #Convert year to datetime
    df['Year'] = pd.to_datetime(df['Year'], format='%Y')

    #Format text columns
    df.columns = [col.replace(' ', '').replace('_', '') for col in df.columns]

    df = df.map(lambda row: row.lower() if isinstance(row, str) else row)


    return df

In [ ]:
# Clean the data
df_car = initial_clean_data(data=df)

print("Initial clean dataframe:")
#df_car.head()

In [ ]:
df_car.shape

In [ ]:
df_car.info()

In [ ]:
#categorical and numerical columns
categorical_columns = []
numerical_columns = []

for col in df_car.columns:
  if df_car[col].dtype == 'object':
    categorical_columns.append(col)
  else:
    numerical_columns.append(col)

In [ ]:
categorical_columns

In [ ]:
numerical_columns

# Data Cleansing

**Handing Missing values**

In [ ]:
# Checking for missing values
missing_values = df_car.isna().sum()
percent_missing_values = (df_car.isna().sum()/df_car.shape[0])*100
missing_values_summary= pd.DataFrame({'missing values': missing_values, 'missing_values (%)': percent_missing_values})
missing_values_summary

#### Column: EngineCylinders

In [ ]:
df_car.loc[(df_car.EngineCylinders.isna())].head()

In [ ]:
df_car.columns

In [ ]:
#Electric cars don't have engine cylinders = 0
df_car.loc[(df_car["TransmissionType"].fillna("") == "direct_drive") &(df_car["EngineFuelType"].fillna("") == "electric"),
    "EngineCylinders"] = 0

In [ ]:
#Mazda RX-7or8 has rotary (wankel) engine hence 0 conventional cylinders
df_car.loc[df_car["Model"].isin(["rx-7", "rx-8"]), "EngineCylinders"] = 0

In [ ]:
df_car.isna().sum()

#### Column: EngineHP

In [ ]:
def fill_engine_hp(df):
    # Fill by similar cars (Make and Model)
    df["EngineHP"] = df.groupby(["Make", "Model"])["EngineHP"].transform(lambda x: x.fillna(x.median()))

    # Fill remaining with median of the make
    df["EngineHP"] = df.groupby("Make")["EngineHP"].transform(lambda x: x.fillna(x.median()))

    # Fill remaining with median

    df_car['EngineHP'] = df_car['EngineHP'].fillna(df_car['EngineHP'].median())

    return df

In [ ]:
# Fill EngineHP
df_car = fill_engine_hp(df_car)

print("Imputing EngineHP:")
df_car.isna().sum()#.sum()

#### Column: NumberofDoors

In [ ]:
df_car.loc[(df_car.NumberofDoors.isna())]

In [ ]:
#Fill by mode of the model of the car
df_car["NumberofDoors"] = df_car.groupby("Model")["NumberofDoors"].transform(lambda x: x.fillna(x.mode()[0] if not x.mode().empty else np.nan))

In [ ]:
df_car.isna().sum()

#### Column: MarketCategory

In [ ]:
df_car.MarketCategory.value_counts()

- percent missing values of Market category is 39% which will significantly reduce the quantity of dataset
- replace missing values of market category based on make or mode and else replace with unknown

In [ ]:
#Fill the missing values of marketcategory based on mode or make
df_car['MarketCategory'] = df_car.groupby(['Make', 'Model'])['MarketCategory'].transform(
    lambda x: x.fillna(x.mode()[0] if not x.mode().empty else 'Unknown')
)

In [ ]:
df_car.isna().sum()

#### Column: EngineFuelType

In [ ]:
df_car['EngineFuelType'] = df_car.groupby('Make')['EngineFuelType'].transform(lambda x: x.fillna(x.mode()[0] if not x.mode().empty else None))

In [ ]:
df_car.isna().sum()

# 2. Feature Engineering

#### Creating new columns: Price per HP (MSRP / Engine HP)

In [ ]:
# No zero divisible error
# df_car['PriceperHP'] = df_car['MSRP'] / df_car['EngineHP']

In [ ]:
# safe handling of potential division by zero error 
df_car['Price_per_HP'] = df_car.apply(
    lambda row: round(row['MSRP'] / row['EngineHP'],2) if row['EngineHP'] > 0 else float('nan'), axis=1)

# summary statistics for the new column: 'PriceperHP'
print("\nMSRP per HP Statistics:")
print(df_car['Price_per_HP'].describe())

print("\nSample of cars with their MSRP per HP:")
print(df_car[['Make', 'Model', 'MSRP', 'EngineHP', 'Price_per_HP']].sample(5))

In [ ]:
df_car.isna().sum()

#### Creating new columns: Total MPG (average of city mpg and highway mpg)

In [ ]:
# Create the new column for total MPG (average of city and highway)
df_car['total_MPG'] = round((df_car['citympg'] + df_car['highwayMPG']) / 2, 2)

# summary statistics for the new column: 'totalMPG'
print("\nTotal MPG Statistics:")
print(df_car['total_MPG'].describe())

print("\nSample of cars with their MPG values:")
print(df_car[['Make', 'Model', 'citympg', 'highwayMPG', 'total_MPG']].sample(5))

print("\nTop 5 cars by total MPG:")
top_mpg = df_car.sort_values('total_MPG', ascending=False) # cars with the best total MPG
print(top_mpg[['Make', 'Model', 'citympg', 'highwayMPG', 'total_MPG']].head(5))

In [ ]:
df_car.isna().sum()

#### Creating new columns: Car Age 

In [ ]:
def new_features(data):
  """
    creates 2 new columns of the DataFrame by:
    - avg_mpg (average of city and highway)
    - price_per_hp (MSRP divided by EngineHP
    - car_age (current year - year)

    Parameters to create new columns:
    df_car.citympg, df_car.highwayMPG, df_car.MSRP and df_car.EngineHP.

    Returns:
    pd.DataFrame: with 2 new columns

    """
  # Create the new column for total MPG (average of city and highway)
  df_car['total_MPG'] = round((df_car['citympg'] + df_car['highwayMPG']) / 2, 2)

  # Dealing of potential zerodivisionerror
  df_car['Price_per_HP'] = df_car.apply(
    lambda row: round(row['MSRP'] / row['EngineHP'],2) if row['EngineHP'] > 0 else float('nan'), axis=1)

  #Depreciation trends
  current_year = 2026

  df_car["Car_Age"] = current_year - df_car["Year"].dt.year

  #Rename some columns
  #df_car.columns = [col.title().replace("Hp", "HP").replace("Mpg", "MPG").replace("Msrp", "MSRP") for col in df_car.columns]

  return df_car

In [ ]:
# new columns
df_car = new_features(data=df_car)

print("New columns in dataframe:")
df_car.head(5)

# 3. Exploratory Data Analysis (EDA)

#### **Descriptive Analysis**

In [ ]:
columns_summary = ['EngineHP', 'MSRP', 'Popularity', 'highwayMPG', 'citympg']

In [ ]:
summary_stats = df_car[columns_summary].describe()
summary_stats

In [ ]:
focused_stats = summary_stats.loc[['mean', '50%', 'std']]
focused_stats = focused_stats.rename(index={'50%': 'median'})
focused_stats.T #transpose the results

#### **Grouped Analysis**

****Checking mean, std and median for some individual parameters****

- VehicleSize
- DrivenWheels
- EngineCylinders

In [ ]:
df_VehicleSize = df_car.groupby(['VehicleSize'])[['MSRP', 'Popularity']].agg(['mean', 'median', 'std'])
df_VehicleSize.columns = ['_'.join(col) for col in df_VehicleSize.columns]
df_VehicleSize = df_VehicleSize.reset_index()
df_VehicleSize

In [ ]:
df_DrivenWheels = df_car.groupby(['DrivenWheels'])[['MSRP', 'Popularity']].agg(['mean', 'median', 'std'])
df_DrivenWheels.columns = ['_'.join(col) for col in df_DrivenWheels.columns]
df_DrivenWheels = df_DrivenWheels.reset_index()
df_DrivenWheels

In [ ]:
df_EngineCylinders = df_car.groupby(['EngineCylinders'])[['MSRP', 'Popularity']].agg(['mean', 'median', 'std'])
df_EngineCylinders.columns = ['_'.join(col) for col in df_EngineCylinders.columns]
df_EngineCylinders = df_EngineCylinders.reset_index()
df_EngineCylinders

****Checking mean, std and median by grouping them****
- VehicleSize
- EngineCylinders
- DrivenWheels

In [ ]:
grouped_stats = df_car.groupby(['DrivenWheels', 'VehicleSize', 'EngineCylinders'])[['MSRP', 'Popularity']].mean()#.reset_index()
grouped_stats.head(10)

In [ ]:
cars_per_group = df_car.groupby(['DrivenWheels', 'VehicleSize', 'EngineCylinders']).size().reset_index(name='car_count')
print("\nNumber of cars in each group:")
print(cars_per_group.head(10))

In [ ]:
# Select the row(s) with maximum car_count
max_count_row = cars_per_group[cars_per_group['car_count'] == cars_per_group['car_count'].max()]

print("\nGroup(s) with the maximum number of cars:")
print(max_count_row)

***Sort by average MSRP to see which combinations are most expensive***

In [ ]:
print("\nGroups sorted by average MSRP (descending):")
print(grouped_stats.sort_values('MSRP', ascending=False).head(10))

***Sort by average Popularity to see which combinations are most popular***

In [ ]:
print("\nGroups sorted by average Popularity (descending):")
print(grouped_stats.sort_values('Popularity', ascending=False).head(10))

***Type of Engine Fuel on the Highway***

In [ ]:
df_car.groupby('EngineFuelType')['highwayMPG'].mean().sort_values(ascending=False)

***Type of Engine Fuel in the city***

In [ ]:
df_car.groupby('EngineFuelType')['citympg'].mean().sort_values(ascending=False)

In [ ]:
df_car.groupby(["TransmissionType"])["EngineHP"].median()

In [ ]:
df_car.columns

#### **Visualizations**

##### A histogram that shows a distribution for the city mpg column.

In [ ]:
sns.histplot(data=df_car, x='citympg')

In [ ]:
sns.histplot(data=df_car, x='highwayMPG')

In [ ]:
# Melt the dataframe to long format for easy faceting
df_mpg = df_car.melt(
    value_vars=['citympg', 'highwayMPG'],
    var_name='MPG_Type',
    value_name='MPG'
)

# Interactive histogram with facets
fig = px.histogram(
    df_mpg,
    x='MPG',
    facet_col='MPG_Type',              # facet by city vs highway
    color_discrete_sequence=['#636EFA'],
    nbins=40,
    title='Distribution of City and Highway MPG',
    labels={'MPG':'Miles per Gallon'}
)

fig.update_layout(showlegend=False)
fig.show()

##### A bar chart showing the average MSRP for each category in Vehicle Size

In [ ]:
# Compute average MSRP per Vehicle Size
avg_msrp = df_car.groupby('VehicleSize')['MSRP'].mean().reset_index()

# Set style
sns.set_style("whitegrid")

# Plot bar chart
plt.figure(figsize=(8,5))
sns.barplot(data=avg_msrp, hue='VehicleSize', y='MSRP', palette='coolwarm')

# Labels and title
plt.xlabel("Vehicle Size")
plt.ylabel("Average MSRP")
plt.title("Average MSRP by Vehicle Size")
plt.show()

##### A scatter plot showing the relationship between Engine HP and MSRP.

In [ ]:
# Create a scatter plot showing the relationship between Engine HP and MSRP
fig = px.scatter(
    df_car,  # Using the original dataframe
    x='EngineHP',
    y='MSRP',
    title='Relationship between Engine Horsepower and MSRP',
    labels={'EngineHP': 'Engine Horsepower', 'MSRP': 'Manufacturer Suggested Retail Price ($)'},
    opacity=0.7,
    color_discrete_sequence=['#636EFA']
)

fig.update_layout(
    xaxis_title='Engine Horsepower (HP)',
    yaxis_title='MSRP ($)'
)

fig.show()

In [ ]:
# Create a scatter plot showing the relationship between Engine HP and MSRP using Seaborn

# Set the style
sns.set_style("whitegrid")

# Create the figure and axes
plt.figure(figsize=(10, 6))

# Create the scatter plot
sns.scatterplot(
    data=df_car,
    x='EngineHP',
    y='MSRP',
    alpha=0.7,
    color='#636EFA')

# Set the title and labels
plt.title('Relationship between Engine Horsepower and MSRP', fontsize=14)
plt.xlabel('Engine Horsepower (HP)', fontsize=12)
plt.ylabel('Manufacturer Suggested Retail Price ($)', fontsize=12)

# Improve the layout
plt.tight_layout()

# Show the plot
plt.show()

##### A boxplot showing the distribution of MSRP for each category in Driven_Wheels.

In [ ]:
# Create a boxplot showing the distribution of MSRP for each category in DrivenWheels

# Set the style
sns.set_style("whitegrid")

# Create the figure with a larger size to accommodate all categories
plt.figure(figsize=(12, 8))

sns.boxplot(data=df_car, hue='DrivenWheels', y='MSRP', palette='coolwarm')
plt.title('Distribution of MSRP by Driven Wheels Type', fontsize=14)
plt.xlabel('Driven Wheels', fontsize=12)
plt.ylabel('Manufacturer Suggested Retail Price ($)', fontsize=12)

plt.xticks(rotation=45)  # Rotate x-axis labels 
plt.tight_layout()   # Improve the layout
plt.show()

In [ ]:
# Create a boxplot showing the distribution of MSRP for each category in DrivenWheels using Plotly

# Create the boxplot
fig = px.box(df_car, x='DrivenWheels', y='MSRP',
    title='Distribution of MSRP by Driven Wheels Type',
    labels={
        'DrivenWheels': 'Driven Wheels',
        'MSRP': 'Manufacturer Suggested Retail Price ($)'},
    color='DrivenWheels',  # Color boxes by category
    category_orders={'DrivenWheels': sorted(df_car['DrivenWheels'].unique())}  # Sort categories alphabetically
)

# Update layout for better appearance
fig.update_layout(
    xaxis_title='Driven Wheels',
    yaxis_title='MSRP ($)',
    showlegend=False,  # Hide legend as it's redundant with x-axis
    height=600,
    width=900
)

# Show the plot
fig.show()

##### Correlation Ananalysis

In [ ]:
df_car['EngineHP'] = pd.to_numeric(df_car['EngineHP'], errors='coerce')

In [ ]:
#df_car.info()

In [ ]:
variables = ['EngineHP', 'MSRP', 'Popularity', 'citympg', 'highwayMPG']
df_corr = df_car[variables].copy()

# Calculate the correlation matrix
corr_matrix = df_corr.corr()

corr_matrix

In [ ]:
# Create a heatmap to visualize correlations
fig = px.imshow(
    corr_matrix,
    text_auto=True,  # Show correlation values
    color_continuous_scale='RdBu_r',  # Red-Blue scale (negative to positive)
    zmin=-1,  # Minimum correlation value
    zmax=1,   # Maximum correlation value
    title='Correlation Matrix: Engine HP, MSRP, Popularity, city mpg, and highway MPG')

# Improve layout
fig.update_layout(
    width=800,
    height=700,
    xaxis_title='',
    yaxis_title='')

# Show the heatmap
fig.show()

In [ ]:
# Investigate correlation between Engine HP, MSRP, Popularity, city mpg, and highway MPG using Seaborn

# Set the style
sns.set(style="white")

# Select the variables of interest
variables = ['EngineHP', 'MSRP', 'Popularity', 'citympg', 'highwayMPG']
df_corr = df_car[variables].copy()

plt.figure(figsize=(10, 8))

corr_matrix = df_corr.corr()

# Create a mask for the upper triangle
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

# Create the heatmap
heatmap = sns.heatmap(
    corr_matrix, 
    annot=True,           # Show correlation values
    fmt=".2f",            # Format as 2 decimal places
    cmap='RdBu_r',        # Red-Blue scale (negative to positive)
    vmin=-1,              # Minimum correlation value
    vmax=1,               # Maximum correlation value
    mask=mask,            # Apply the mask to show only lower triangle
    linewidths=0.5,       # Add lines between cells
    cbar_kws={"shrink": .8}
)

# Set title
plt.title('Correlation Matrix: Engine HP, MSRP, Popularity, city mpg, and highway MPG', fontsize=14)
plt.tight_layout()

# Show the heatmap
plt.show()